# Topic 4 — Multi-Agent Conflict Resolution (LangGraph, Azure OpenAI)

Self-contained Colab port of `topic-4-multi-agent-conflict-resolution/` from
the `project-2026-01-claude` repo, as of 2026-09-13.

**Architecture** (unchanged from the source project — see `docs/component-specs.md`):
- A real `langgraph.graph.StateGraph` (not hand-rolled) — `Start → Planning → Audit
  → Decision Node`, looping back to Planning on a failed audit, per B7/B8.
- **Agent A (SEO Content Planner)** and **Agent B (YMYL Compliance Auditor)** —
  both dynamic, both backed by Azure OpenAI (Planner temp 0.7, Auditor temp
  0.1 — see component-specs.md's run notes for why they differ).
- **Fault tolerance (B19)** — every LLM call retries up to 3 times on
  transient errors (rate limit / timeout / connection) with 1s/2s backoff;
  a malformed schema response gets a single validation retry, then a clear,
  named failure — never a silent pass-through.
- **B12's graph visualization is captured directly from the compiled graph's
  own introspection** (`get_graph().draw_mermaid()`), not hand-drawn.

**A real, checked finding**: `data/Manual.txt` (the "company's internal
manual" Agent B is supposed to enforce) is entirely mortgage/interest-rate
-specific — it has no casino-related rule at all, so it can't be the source
of this scenario's forbidden terms (穩賺不賠/保證出金). Those come from B10's
own fixture, used as an explicit rule, not force-fit from a document that
doesn't cover this content category. See `docs/component-specs.md` run notes.

**Two demo cells near the end**: a **Fixture dry run** (no API key needed —
step through the graph's actual mechanics first, both the converged and
exhausted paths) and a **real run against Azure OpenAI** (B9/B10's actual
forced-conflict scenario).

**Before running the real-Azure cell:** open the 🔑 Secrets panel in Colab's
left sidebar and add four secrets, then grant this notebook access to each:
`AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_VERSION`,
`AZURE_OPENAI_DEPLOYMENT`.

In [ ]:
!pip install -q "langgraph>=0.2.0" "pydantic>=2.0.0" "openai>=1.40.0" "python-dotenv>=1.0.0"


## 1. Credentials — read from Colab Secrets, never typed or saved to disk

In [ ]:
REQUIRED_SECRETS = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT",
]

for _name in REQUIRED_SECRETS:
    try:
        os.environ[_name] = userdata.get(_name)
    except userdata.SecretNotFoundError:
        raise RuntimeError(
            f"Colab secret '{_name}' isn't set. Add it via the \N{KEY} Secrets "
            "panel in the left sidebar, then re-run this cell."
        ) from None
    except userdata.NotebookAccessError:
        raise RuntimeError(
            f"Colab secret '{_name}' exists but this notebook hasn't been "
            "granted access yet -- toggle notebook access on for it in the "
            "Secrets panel, then re-run this cell."
        ) from None

print("Loaded Azure OpenAI config from Colab secrets:", ", ".join(REQUIRED_SECRETS))


## 2. Shared imports

Every name every cell below needs, in one place — the fix topic 2's own postmortem asked for: this notebook is executed end-to-end (via a Fixture backend) as part of building it, not just syntax-checked, so a name missing from this cell fails immediately here rather than surfacing later during a live run.

In [ ]:
import asyncio
import json
import os
import re
from typing import Awaitable, Callable, Dict, List, Literal, Optional, Protocol, TypedDict, TypeVar

from dotenv import load_dotenv
from google.colab import userdata
from pydantic import BaseModel, ValidationError, field_validator, model_validator
from langgraph.graph import END, START, StateGraph

load_dotenv()  # no-op in Colab (no .env file) -- kept for parity with the real llm_client.py


## 3. Data — `data/Manual.txt`, embedded verbatim

Colab has no access to your local `data/` folder, so its real content is embedded below as a string, passed directly into `initial_state(...)` — `src/graph.py` takes `compliance_reference` as a parameter rather than resolving a path internally (see its docstring for why), so no file I/O or path juggling is needed here at all.

In [ ]:
MANUAL_TEXT = '關鍵要求：所有關於利率的描述，必須標註「需視個人信用條件而定」。\nEEAT 規範：必須提及「銀行」與「代書/民間」二胎的法律權益差異。\n禁忌：嚴禁出現「保證過件」、「全台最低利」等誇大字眼。'
print(MANUAL_TEXT)


## 4. State schema + output schemas (`src/schemas.py`)

In [ ]:
"""Data shapes for topic 4: the two agents' validated output (G2/G3's
output schemas in docs/component-specs.md) and the LangGraph state itself
(G1). Pydantic models validate each LLM call's output before it's allowed
into the graph's state; the state itself is a plain TypedDict, which is
what langgraph.graph.StateGraph expects and merges node return values into.
"""


class PlannerOutput(BaseModel):
    """Agent A's (SEO Content Planner) validated output. `outline` and
    `keywords_used` must be non-empty — an empty outline or a draft that
    claims to use no keywords at all isn't a real attempt at the brief,
    it's a malformed response that happens to parse as valid JSON."""

    outline: List[str]
    keywords_used: List[str]
    rationale: str

    @field_validator("outline", "keywords_used")
    @classmethod
    def _non_empty_list(cls, v: List[str], info) -> List[str]:
        if not v:
            raise ValueError(f"{info.field_name} must not be empty")
        return v

    @field_validator("rationale")
    @classmethod
    def _non_empty_str(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("rationale must not be empty")
        return v


class AuditVerdict(BaseModel):
    """Agent B's (YMYL Compliance Auditor) validated output. `issues` and
    `revision_suggestions` must be empty exactly when `passed` is true and
    non-empty exactly when it's false — a fail with no stated reason, or a
    pass with lingering issues, is a malformed response, not a valid edge
    case (docs/component-specs.md's G3 spec)."""

    passed: bool
    issues: List[str]
    revision_suggestions: List[str]

    @model_validator(mode="after")
    def _issues_match_passed(self) -> "AuditVerdict":
        if self.passed and (self.issues or self.revision_suggestions):
            raise ValueError("passed=true must not carry issues or revision_suggestions")
        if not self.passed and not (self.issues and self.revision_suggestions):
            raise ValueError("passed=false must carry non-empty issues and revision_suggestions")
        return self


class HistoryEntry(TypedDict):
    """One completed round, kept so a later Planner call can see what it
    already tried and why it was rejected — the concrete mechanism behind
    B17's "does State fully carry prior context" criterion."""

    iteration: int
    outline: dict  # PlannerOutput.model_dump()
    audit_verdict: dict  # AuditVerdict.model_dump()


class NegotiationState(TypedDict):
    """The full LangGraph state (G1). Every field a node reads or writes
    is named here explicitly — langgraph merges each node's returned dict
    into this state, so this is the one place the whole shape is visible
    at a glance."""

    keyword: str
    category: str
    required_ctr_terms: List[str]
    forbidden_terms: List[str]
    compliance_reference: str
    outline: Optional[dict]  # PlannerOutput.model_dump(), current round
    audit_verdict: Optional[dict]  # AuditVerdict.model_dump(), current round
    iteration: int
    max_iterations: int
    history: List[HistoryEntry]
    status: Literal["in_progress", "converged", "exhausted"]


## 5. Defensive JSON extraction (`src/json_utils.py`)

In [ ]:
"""Defensive JSON extraction — same approach as topic 3's notebook (strip
markdown fences, find the first balanced {...} by bracket-counting, never a
bare json.loads). Azure's JSON mode makes this less likely to be needed
than topic 3's un-tuned local model, but the two agents here are also
freer-form than topic 2's tightly-templated prompts, so it's kept rather
than assumed unnecessary.
"""


def extract_json(raw: str) -> dict:
    """Raises json.JSONDecodeError (via the final json.loads) if no valid
    JSON object can be found at all — callers decide what that means
    (topic 4's nodes treat it as a validation failure, see src/graph.py)."""
    text = raw.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    start = text.find("{")
    if start == -1:
        return json.loads(text)  # will raise, with a clear message

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start : i + 1])
    return json.loads(text[start:])  # unbalanced -- will raise


## 6. Fault tolerance — async retry with backoff (`src/retry.py`, B19)

In [ ]:
"""Fault tolerance for the two dynamic nodes (B19, docs/component-specs.md's
run notes): a real async retry with exponential backoff, scoped specifically
to transient API errors, not a blanket catch-and-retry-anything.

Two genuinely different failure modes are handled differently on purpose:
- Transient (rate limit / timeout / connection): retried here, silently
  recovered if a later attempt succeeds — the caller never sees it.
- Malformed output (bad JSON, failed schema validation): NOT retried by
  this function. A schema failure isn't fixed by resending the identical
  prompt to a model that just failed to follow it; src/graph.py retries
  that case once with the same input (per component-specs.md's G2/G3
  spec), then surfaces it as a real, named failure.
"""


T = TypeVar("T")

ATTEMPTS = 3
BACKOFF_SECONDS = (1, 2)  # delay before the 2nd and 3rd attempts — see component-specs.md run notes


async def call_with_retry(fn: Callable[[], Awaitable[T]]) -> T:
    """Retries `fn` (a zero-arg async callable — callers wrap their real
    call in a lambda/closure) on transient OpenAI/Azure errors: 3 attempts
    total, waiting 1s then 2s between them. Raises the last transient
    error if every attempt is exhausted; any other exception propagates
    immediately, unretried."""
    from openai import APIConnectionError, APITimeoutError, RateLimitError

    last_exc: Exception | None = None
    for attempt in range(ATTEMPTS):
        if attempt > 0:
            await asyncio.sleep(BACKOFF_SECONDS[attempt - 1])
        try:
            return await fn()
        except (RateLimitError, APITimeoutError, APIConnectionError) as e:
            last_exc = e
            continue
    raise RuntimeError(
        f"LLM call failed after {ATTEMPTS} attempts (transient errors): {last_exc}"
    ) from last_exc


## 7. Prompt templates + rendering (`prompts/*.md`, `src/prompts.py`)

The two templates below are embedded **verbatim** from `topic-4-multi-agent-conflict-resolution/prompts/*.md`. The rendering functions are ported verbatim from `src/prompts.py`, with the file-reading `_load(...)` calls swapped for references to the embedded constants.

In [ ]:
PLANNER_TEMPLATE = '<!--\nTraceability: prompts/master-prompt.md → build-topic-executables, {#}=4, component G2\nSource: topic-4-multi-agent-conflict-resolution/docs/problem-statement-categorized.md\n  Role: B4\n  Goal: B2, B11\nType: dynamic\n\nRendered twice per graph run in different modes: round 0 with no revision\nfeedback (the maximally CTR-driven first draft the B10 fixture expects to\nfail audit), and round N>0 with the prior round\'s rejection reason plus\nfull history -- see docs/component-specs.md\'s G2 spec for exactly what\'s\ntemplated into each.\n-->\n\n# SEO Content Planner (Agent A)\n\n## Role\nYou are the SEO Content Planner: your job is to maximize search\nclick-through rate. You prefer eye-catching, highly competitive keywords\nand titles, and you push for the version of any content that will get the\nmost clicks. (B4)\n\n## Goal\nGiven a target keyword and category, produce a content outline aimed at\nmaximizing click-through rate. If a prior round\'s outline was rejected by\nthe Compliance Auditor, revise it to address the *specific* issues raised\n— without abandoning the click-through goal just because one round was\nrejected. (B2, B11)\n\n## Instructions\n- **Constraints:**\n  - You are specifically instructed to lean on these high-CTR terms for\n    this keyword, because they perform well: `{required_ctr_terms}` (B10\'s\n    fixture — this is not a hypothetical preference, it\'s how this run is\n    configured).\n  - On a revision round, you will be given the prior outline, the specific\n    reasons it was rejected, and what to fix. **Address exactly those\n    reasons.** Do not simply delete every aggressive claim wholesale —\n    that trades one failure mode (over-promising) for another\n    (one-sidedly conceding everything, which B18 explicitly checks for).\n    Find the version of the claim that\'s still compelling *and* accurate\n    — e.g. an honest win-rate disclosure or a security/audit-compliance\n    angle can be just as clickable as an unqualified guarantee, if you\n    commit to making it specific and concrete rather than vague.\n  - Do not repeat a specific phrasing this brief\'s history already shows\n    was rejected for the same reason — if `{history}` shows a term was\n    already rejected, the revision has to actually change, not resubmit\n    the same draft with cosmetic edits.\n- **Reference:**\n  - `{keyword}`, `{category}` — the content this round is planning for.\n  - `{required_ctr_terms}` — the high-CTR terms you\'re configured to favor.\n  - `{revision_suggestions}` (round 0: none) — the prior round\'s specific,\n    actionable feedback from the Compliance Auditor, if this is a revision.\n  - `{history}` — every prior round\'s outline and verdict in this run, so\n    you don\'t repeat a rejected approach.\n- **Output shape** (feeds G4\'s routing and G6\'s rendering):\n  - `{"outline": [string, ...], "keywords_used": [string, ...], "rationale": string}`.\n  - `outline` and `keywords_used` must be non-empty.\n  - `rationale` states specifically why this draft should perform well on\n    click-through — not a generic "this is engaging" claim.\n- **Self-check before finalizing** (from B11, B18):\n  - If this is a revision, does it demonstrably address every issue the\n    Auditor raised last round — not just some of them?\n  - Does this draft still have a real, specific hook for click-through, or\n    did I just strip everything the Auditor flagged and leave something\n    bland? (A bland-but-safe outline is not what B11/B18 are asking for.)\n  - Is the output valid JSON in exactly the required shape?\n'

AUDITOR_TEMPLATE = '<!--\nTraceability: prompts/master-prompt.md → build-topic-executables, {#}=4, component G3\nSource: topic-4-multi-agent-conflict-resolution/docs/problem-statement-categorized.md\n  Role: B5\n  Goal: B2, B11\nType: dynamic\n\nManual.txt is loaded as static reference text (see docs/component-specs.md\nrun notes for why this isn\'t a RAG/retrieval step). It\'s entirely\nmortgage/interest-rate-specific and has no casino-related rule at all --\nthis prompt says so explicitly rather than forcing a citation that doesn\'t\napply, and treats forbidden_terms (B10\'s own fixture) as the operative\nrule for casino content.\n-->\n\n# YMYL Compliance Auditor (Agent B)\n\n## Role\nYou are the YMYL Compliance Auditor: your job is to ensure content is\n100% compliant with legal regulations and the company\'s internal manual.\nYou have zero tolerance for exaggerated, misleading, or high-risk wording\n— you are not here to negotiate the Planner\'s click-through goals down,\nonly to enforce what compliance actually requires. (B5)\n\n## Goal\nGiven a content outline, determine whether it passes compliance review. If\nit doesn\'t, give specific, constructive revision suggestions the Planner\ncan actually act on — not just a restatement of which rule was broken.\n(B2, B11)\n\n## Instructions\n- **Constraints:**\n  - **Blocklist check (hard fail, zero tolerance):** if the outline uses\n    any of `{forbidden_terms}` (or a close paraphrase with the same\n    unqualified-guarantee meaning), it fails, full stop — this is the\n    specific conflict this scenario is built to test (B10).\n  - **Manual.txt check (`{compliance_reference}`), only where it applies:**\n    Manual.txt\'s rules are about interest-rate/mortgage content\n    specifically (an individual-credit-conditions disclaimer on rate\n    claims, disclosing the bank-vs-private-lender legal distinction,\n    banning "guaranteed approval"/"lowest rate nationwide" language). If\n    this outline is about a different category (e.g. Casino), say\n    explicitly that Manual.txt doesn\'t apply to this content rather than\n    forcing an irrelevant citation — don\'t invent a rule Manual.txt\n    doesn\'t actually contain for this category.\n  - **Every `revision_suggestions` entry must name a concrete alternative**,\n    not just the violation — e.g. "replace the guaranteed-payout claim with\n    an honest disclosure of the actual win rate" or "reframe around\n    security/audit-compliance instead of a payout guarantee," not just\n    "remove the guarantee language" (B18 grades whether your suggestions\n    are constructive enough to actually guide a fix).\n  - You are auditing the outline as given — you do not rewrite it\n    yourself; that\'s the Planner\'s job on the next round.\n- **Reference:**\n  - `{outline}`, `{keywords_used}` — this round\'s draft from the Planner.\n  - `{forbidden_terms}` — the hard blocklist for this scenario (B10).\n  - `{compliance_reference}` — Manual.txt\'s full text (may not apply to\n    this content\'s category; say so if not).\n- **Output shape** (feeds G4\'s routing and G6\'s rendering):\n  - `{"passed": bool, "issues": [string, ...], "revision_suggestions": [string, ...]}`.\n  - `issues` and `revision_suggestions` must both be empty if `passed` is\n    `true`, and both non-empty if `passed` is `false` — a fail with no\n    stated reason, or a pass with lingering issues, is itself a malformed\n    response.\n- **Self-check before finalizing** (from B18):\n  - Did I check the actual blocklist terms, not just my general impression\n    of whether the tone "feels" compliant?\n  - Does every `revision_suggestions` entry give the Planner something\n    concrete to do, not just repeat which rule was broken?\n  - Is the output valid JSON in exactly the required shape?\n'

"""Loads the Role/Goal/Instructions prompt files for topic 4 and renders
them with the runtime state each node needs. Source templates:
../prompts/planner.md, ../prompts/auditor.md.

Same convention as topic 1's prompts.py: the template's own prose
placeholders like {keyword} are plain string-replaced (safe — they're
single words, no risk of colliding with the Output shape section's
literal JSON braces), while structured values (history, required terms,
the current outline) are appended as a labeled runtime-data block after
the template rather than substituted inline — str.format() would choke on
the template's own literal `{"outline": [...], ...}` example.
"""




def _json(value) -> str:
    return json.dumps(value, ensure_ascii=False, indent=2)


def planner_prompt(state: NegotiationState) -> str:
    template = PLANNER_TEMPLATE
    template = template.replace("{keyword}", state["keyword"]).replace("{category}", state["category"])

    lines = [
        template,
        "\n---",
        f"Keyword: {state['keyword']}   Category: {state['category']}",
        "Required high-CTR terms for this run:",
        _json(state["required_ctr_terms"]),
    ]
    if state["iteration"] == 0:
        lines.append("\nThis is round 0 — no prior feedback yet, no history to avoid repeating.")
    else:
        last = state["history"][-1]
        lines.append(f"\nThis is round {state['iteration']} — a revision.")
        lines.append("Prior round's revision suggestions to address:")
        lines.append(_json(last["audit_verdict"]["revision_suggestions"]))
        lines.append("\nFull history so far (do not repeat a rejected approach):")
        lines.append(_json(state["history"]))
    lines.append(
        "\nRespond with ONLY a JSON object: "
        '{"outline": [string, ...], "keywords_used": [string, ...], "rationale": string}.'
    )
    return "\n".join(lines)


def auditor_prompt(state: NegotiationState) -> str:
    template = AUDITOR_TEMPLATE

    lines = [
        template,
        "\n---",
        f"Category: {state['category']}",
        "Outline to audit:",
        _json(state["outline"]),
        "\nForbidden terms for this run (hard fail, zero tolerance):",
        _json(state["forbidden_terms"]),
        "\nCompliance manual (may not apply to this category — say so if not):",
        state["compliance_reference"],
        "\nRespond with ONLY a JSON object: "
        '{"passed": bool, "issues": [string, ...], "revision_suggestions": [string, ...]}.',
    ]
    return "\n".join(lines)


## 8. LLM client — Azure OpenAI + Fixture (`src/llm_client.py`)

In [ ]:
"""Pluggable LLM client for topic 4 — same Protocol-plus-Fixture pattern as
topics 1-2's llm_client.py, made async this time. Topic 4 is the first
topic where "asynchronous calls handled correctly" is itself an evaluated
requirement (B19), so this uses the real AsyncAzureOpenAI client rather
than wrapping a sync call in a thread — a genuinely async network call,
not a fake-async shim around blocking I/O.

No EmbeddingClient here: topic 4 doesn't do retrieval (see
docs/component-specs.md's run notes for why Manual.txt is loaded as static
text instead).
"""


load_dotenv()


class LLMClient(Protocol):
    async def complete(self, prompt: str, *, component: str) -> str: ...


def _require_env(names: list[str]) -> Dict[str, str]:
    """Same validation as topics 1-2: every named var must be present and
    not an obvious .env.example placeholder."""
    values: Dict[str, str] = {}
    missing = []
    for name in names:
        val = os.environ.get(name, "")
        if not val or val.startswith("your-") or "<" in val:
            missing.append(name)
        else:
            values[name] = val
    if missing:
        raise RuntimeError("Missing real .env values for: " + ", ".join(missing))
    return values


class AzureOpenAILLMClient:
    """Real backend for both agents (Planner temp 0.7, Auditor temp 0.1 —
    see docs/component-specs.md's run notes for why they differ). One
    client instance is reused across both agents and every round; a fresh
    AsyncAzureOpenAI is constructed per call rather than held open across
    the whole graph run, same trade-off topics 1-2 made (simplicity over
    connection reuse) — not a bottleneck at this call volume."""

    def __init__(self) -> None:
        vals = _require_env([
            "AZURE_OPENAI_API_KEY",
            "AZURE_OPENAI_ENDPOINT",
            "AZURE_OPENAI_API_VERSION",
            "AZURE_OPENAI_DEPLOYMENT",
        ])
        self._key = vals["AZURE_OPENAI_API_KEY"]
        self._endpoint = vals["AZURE_OPENAI_ENDPOINT"]
        self._api_version = vals["AZURE_OPENAI_API_VERSION"]
        self._deployment = vals["AZURE_OPENAI_DEPLOYMENT"]

    async def complete(self, prompt: str, *, component: str) -> str:
        from openai import AsyncAzureOpenAI  # imported lazily so this module loads without the package installed

        temperature = 0.7 if component == "planner" else 0.1
        client = AsyncAzureOpenAI(
            api_key=self._key,
            azure_endpoint=self._endpoint,
            api_version=self._api_version,
        )
        response = await client.chat.completions.create(
            model=self._deployment,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=2048,  # not max_tokens — see topic 1's llm_client.py
            temperature=temperature,
            response_format={"type": "json_object"},
        )
        return response.choices[0].message.content


class FixtureLLMClient:
    """Same queued-response pattern as topics 1-2's fixtures, made async.
    Queued per component ('planner' / 'auditor'), consumed in order — so a
    multi-round fixture run needs one queued response per expected call."""

    def __init__(self, fixtures: Dict[str, list]) -> None:
        self._fixtures = {k: list(v) for k, v in fixtures.items()}
        self.calls: list[str] = []

    async def complete(self, prompt: str, *, component: str) -> str:
        self.calls.append(component)
        queue = self._fixtures.get(component)
        if not queue:
            raise KeyError(f"No fixture left for component '{component}'")
        return queue.pop(0)


## 9. LangGraph wiring — the actual `StateGraph` (`src/graph.py`)

In [ ]:
"""The actual LangGraph wiring (G4, G5 in docs/component-specs.md) plus the
two dynamic nodes (G2, G3). Real langgraph.graph.StateGraph, not a
hand-rolled equivalent — so B12's graph visualization comes from the
compiled graph's own introspection (see render.py), not a hand-drawn
diagram that could silently drift from what this file actually wires.
"""


def initial_state(
    *,
    keyword: str,
    category: str,
    required_ctr_terms: list[str],
    forbidden_terms: list[str],
    compliance_reference: str,
    max_iterations: int = 5,  # engineering default, see component-specs.md run notes
) -> NegotiationState:
    """`compliance_reference` (Manual.txt's text) is loaded by the caller,
    not resolved internally via a hardcoded path — same reasoning
    topic 2's `ingest_manual(path, ...)` and `load_serp_data(path)` already
    follow: this module stays portable (verbatim-portable into a Colab
    notebook cell, with no `__file__`-relative path to break) and testable
    without touching the real filesystem."""
    return NegotiationState(
        keyword=keyword,
        category=category,
        required_ctr_terms=required_ctr_terms,
        forbidden_terms=forbidden_terms,
        compliance_reference=compliance_reference,
        outline=None,
        audit_verdict=None,
        iteration=0,
        max_iterations=max_iterations,
        history=[],
        status="in_progress",
    )


async def _call_and_validate(llm: LLMClient, prompt: str, component: str, model_cls):
    """Shared logic for both dynamic nodes: call the LLM (with transient-
    error retry), parse defensively, validate against the schema. A
    validation failure (malformed JSON, or JSON that doesn't satisfy the
    schema) is retried exactly once against the same prompt — per
    docs/component-specs.md's G2/G3 spec, this is a different failure mode
    than a transient network error and gets a different, much smaller
    retry budget. If both attempts fail, raise a clear, named error rather
    than let a malformed value silently enter the graph's state."""
    last_error: Exception | None = None
    for validation_attempt in range(2):  # one real attempt + one retry
        raw = await call_with_retry(lambda: llm.complete(prompt, component=component))
        try:
            parsed = extract_json(raw)
            return model_cls.model_validate(parsed)
        except (ValidationError, ValueError) as e:
            last_error = e
            continue
    raise RuntimeError(
        f"{component} produced invalid output after retrying validation once: {last_error}\n"
        f"Last raw response: {raw!r}"
    )


def make_planner_node(llm: LLMClient):
    async def planner_node(state: NegotiationState) -> dict:
        prompt = planner_prompt(state)
        output = await _call_and_validate(llm, prompt, "planner", PlannerOutput)
        return {"outline": output.model_dump()}

    return planner_node


def make_auditor_node(llm: LLMClient):
    async def auditor_node(state: NegotiationState) -> dict:
        prompt = auditor_prompt(state)
        verdict = await _call_and_validate(llm, prompt, "auditor", AuditVerdict)
        return {"audit_verdict": verdict.model_dump()}

    return auditor_node


def decide_node(state: NegotiationState) -> dict:
    """G4's deterministic bookkeeping half: appends this round to history
    (if it's not converging) and sets the terminal status. Kept as its
    own node — separate from the routing function below — because
    updating state and choosing the next node are two different jobs;
    langgraph conditional-edge routing functions should only read state,
    not mutate it."""
    verdict = state["audit_verdict"]
    if verdict["passed"]:
        return {"status": "converged"}

    if state["iteration"] + 1 >= state["max_iterations"]:
        # The outline being returned here is still the one this exact
        # verdict just evaluated and rejected — status says so plainly,
        # no bonus audit call needed (component-specs.md's G4 spec).
        return {"status": "exhausted"}

    entry: HistoryEntry = {
        "iteration": state["iteration"],
        "outline": state["outline"],
        "audit_verdict": verdict,
    }
    return {
        "history": state["history"] + [entry],
        "iteration": state["iteration"] + 1,
        "status": "in_progress",
    }


def route_after_decide(state: NegotiationState) -> Literal["planner", "end"]:
    """The actual conditional-edge routing function — reads the status
    decide_node just set and nothing else."""
    return "end" if state["status"] in ("converged", "exhausted") else "planner"


def build_graph(llm: LLMClient):
    """Wires Start -> Planning -> Audit -> Decision Node exactly as B7
    names it, with the conditional edge B8 describes."""
    graph = StateGraph(NegotiationState)
    graph.add_node("planner", make_planner_node(llm))
    graph.add_node("auditor", make_auditor_node(llm))
    graph.add_node("decide", decide_node)

    graph.add_edge(START, "planner")
    graph.add_edge("planner", "auditor")
    graph.add_edge("auditor", "decide")
    graph.add_conditional_edges("decide", route_after_decide, {"planner": "planner", "end": END})

    return graph.compile()


## 10. Output rendering — the three B12/B13/B16 deliverables (`src/render.py`)

In [ ]:
"""G6 — Output Renderer: turns a finished graph run into the three
deliverables B12/B13/B16 actually ask for. Pure formatting over an
already-final NegotiationState; no new judgment happens here.
"""


def render_graph_diagram(compiled_graph) -> str:
    """B12 — captured directly from the compiled LangGraph object's own
    introspection, not hand-drawn, so it can't drift from what the code
    actually wires."""
    return compiled_graph.get_graph().draw_mermaid()


def render_state_log(state: NegotiationState) -> str:
    """B13 — a full per-round transcript: every prior round from
    `history`, plus the final round (whichever ended the run)."""
    lines: list[str] = ["# State Log — full negotiation transcript", ""]

    rounds = list(state["history"])
    final_round_num = state["history"][-1]["iteration"] + 1 if state["history"] else 0
    rounds.append({
        "iteration": final_round_num,
        "outline": state["outline"],
        "audit_verdict": state["audit_verdict"],
    })

    for r in rounds:
        outline = r["outline"]
        verdict = r["audit_verdict"]
        lines.append(f"## Round {r['iteration']}")
        lines.append("")
        lines.append(f"**Planner (Agent A) outline** — keywords used: {', '.join(outline['keywords_used'])}")
        for item in outline["outline"]:
            lines.append(f"- {item}")
        lines.append(f"\n*Rationale:* {outline['rationale']}")
        lines.append("")
        verdict_word = "PASSED" if verdict["passed"] else "REJECTED"
        lines.append(f"**Auditor (Agent B) verdict: {verdict_word}**")
        if verdict["issues"]:
            lines.append("Issues:")
            for issue in verdict["issues"]:
                lines.append(f"- {issue}")
        if verdict["revision_suggestions"]:
            lines.append("Revision suggestions:")
            for s in verdict["revision_suggestions"]:
                lines.append(f"- {s}")
        lines.append("")

    return "\n".join(lines)


def render_final_output(state: NegotiationState) -> str:
    """B16 — the final outline plus a plain-language summary of the
    negotiation's key turning points. `status` is reported as-is
    ("converged" or "exhausted") — never smoothed into an implied success."""
    outline = state["outline"]
    lines = ["# Final Output", ""]
    lines.append(f"**Status: {state['status']}** (after {state['iteration'] + 1} round(s))")
    lines.append("")
    lines.append("## Final article outline")
    for item in outline["outline"]:
        lines.append(f"- {item}")
    lines.append(f"\n**Keywords used:** {', '.join(outline['keywords_used'])}")
    lines.append(f"\n**Rationale:** {outline['rationale']}")
    lines.append("")

    lines.append("## Negotiation summary")
    if not state["history"] and state["status"] == "converged":
        lines.append("Converged on round 0 — the first draft passed audit with no revision needed.")
    else:
        rejected_rounds = list(state["history"])
        if state["status"] == "exhausted":
            # The round that triggered "exhausted" was never appended to
            # history (see graph.py's decide_node) — include it here so
            # the summary doesn't silently omit the final rejection.
            rejected_rounds = rejected_rounds + [{
                "iteration": state["iteration"],
                "audit_verdict": state["audit_verdict"],
            }]
        lines.append(f"Took {len(rejected_rounds)} rejected round(s) before {state['status']}:")
        for entry in rejected_rounds:
            v = entry["audit_verdict"]
            lines.append(
                f"- Round {entry['iteration']}: rejected for "
                f"{'; '.join(v['issues'])} → asked to {'; '.join(v['revision_suggestions'])}"
            )
        if state["status"] == "exhausted":
            lines.append(
                f"\n**Did not converge within {state['max_iterations']} rounds** — the "
                "outline above is the last-audited version, still carrying the "
                "Auditor's final unresolved issues, not a passed result."
            )

    return "\n".join(lines)


## 11. Dry run with a scripted Fixture — no API key needed

Steps through the graph's actual mechanics first: round 0 is scripted to fail (B10's forced conflict — the Planner's draft leans on the forbidden terms), round 1 is scripted to pass. Exercises the real `build_graph`, real `langgraph` routing, and the real render functions — only the LLM responses are canned.

In [ ]:
_round0_planner = json.dumps({
    "outline": ["標題：娛樂城推薦 - 穩賺不賠的選擇", "段落一：介紹平台", "段落二：保證出金流程"],
    "keywords_used": ["穩賺不賠", "保證出金"],
    "rationale": "高點擊率的保證用語能吸引急於獲利的使用者點擊。",
})
_round0_audit = json.dumps({
    "passed": False,
    "issues": ["包含禁用詞「穩賺不賠」", "包含禁用詞「保證出金」"],
    "revision_suggestions": ["以誠實揭露勝率取代「穩賺不賠」", "以強調資安審查取代「保證出金」"],
})
_round1_planner = json.dumps({
    "outline": ["標題：娛樂城推薦 - 誠實勝率揭露與資安保障", "段落一：介紹平台", "段落二：資安審查機制"],
    "keywords_used": ["誠實勝率揭露", "資安審查"],
    "rationale": "誠實揭露勝率與強調資安審查仍具吸引力，且符合合規要求。",
})
_round1_audit = json.dumps({"passed": True, "issues": [], "revision_suggestions": []})

fixture_llm = FixtureLLMClient({
    "planner": [_round0_planner, _round1_planner],
    "auditor": [_round0_audit, _round1_audit],
})
fixture_graph = build_graph(fixture_llm)
fixture_state = initial_state(
    keyword="娛樂城推薦",
    category="Casino",
    required_ctr_terms=["穩賺不賠", "保證出金"],
    forbidden_terms=["穩賺不賠", "保證出金"],
    compliance_reference=MANUAL_TEXT,
    max_iterations=5,
)
fixture_final_state = await fixture_graph.ainvoke(fixture_state)

print("=== Graph diagram (B12) ===")
print(render_graph_diagram(fixture_graph))
print("\n=== State log (B13) ===")
print(render_state_log(fixture_final_state))
print("\n=== Final output (B16) ===")
print(render_final_output(fixture_final_state))


## 12. Real run against Azure OpenAI — B9/B10's actual scenario

Colab supports top-level `await` directly in a code cell — no `asyncio.run()` needed. Change `KEYWORD`/`CATEGORY`/the term lists below to try a different scenario; the defaults reproduce B9/B10 exactly.

In [ ]:
KEYWORD = "娛樂城推薦"
CATEGORY = "Casino"
REQUIRED_CTR_TERMS = ["穩賺不賠", "保證出金"]
FORBIDDEN_TERMS = ["穩賺不賠", "保證出金"]
MAX_ITERATIONS = 5

azure_llm = AzureOpenAILLMClient()
graph = build_graph(azure_llm)
state = initial_state(
    keyword=KEYWORD,
    category=CATEGORY,
    required_ctr_terms=REQUIRED_CTR_TERMS,
    forbidden_terms=FORBIDDEN_TERMS,
    compliance_reference=MANUAL_TEXT,
    max_iterations=MAX_ITERATIONS,
)
final_state = await graph.ainvoke(state)

print("=== Graph diagram (B12) ===")
print(render_graph_diagram(graph))
print("\n=== State log (B13) ===")
print(render_state_log(final_state))
print("\n=== Final output (B16) ===")
print(render_final_output(final_state))


## 13. Next steps

- Try a different keyword/category (e.g. a mortgage scenario) to see whether Agent B correctly says Manual.txt's rules don't apply to a Casino scenario, and correctly applies them when they do.
- Lower `MAX_ITERATIONS` to see the `exhausted` (non-convergence) path for real against a live model, not just the fixture.
- The full per-round transcript above is exactly what `docs/component-specs.md`'s B13 (State Logs) deliverable asks for — save this cell's output for the record.